In [11]:
import dlib
import numpy as np
import cv2
import os
from skimage.io import imread_collection
import tensorflow as tf
from tensorflow.keras import layers as KL
from tensorflow.keras import models as KM
from tensorflow.keras import backend as K
from tensorflow.keras import Model

# Declaring Supporting classes for importing tensorflow model

In [12]:
class ScaleLayer(KL.Layer):
    def __init__(self, **kwargs):
        super(ScaleLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) >= 2

        if K.image_data_format() == 'channels_last':
            ndim = int(input_shape[-1])
        else:
            ndim = int(input_shape[1])

        self.gamma = self.add_weight(name='gamma', shape=(ndim, ))
        self.beta = self.add_weight(name='beta', shape=(ndim, ))

        super(ScaleLayer, self).build(input_shape)

    def call(self, x):
        input_shape = K.int_shape(x)

        bn_axis = 3 if K.image_data_format() == 'channels_last' else 1

        broadcast_shape = [1] * len(input_shape)
        broadcast_shape[bn_axis] = input_shape[bn_axis]

        broadcast_gamma = K.reshape(self.gamma, broadcast_shape)
        broadcast_beta = K.reshape(self.beta, broadcast_shape)

        output = tf.math.multiply(x, broadcast_gamma)
        output = tf.math.add(output, broadcast_beta)
        return output

    def compute_output_shape(self, input_shape):
        return input_shape
        
class ReshapeLayer(KL.Layer):
    def __init__(self, **kwargs):
        super(ReshapeLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) >= 2
        super(ReshapeLayer, self).build(input_shape)

    def call(self, x):
        s = K.shape(x)
        zeros_w = tf.zeros((s[0], 1, s[2], s[3]), tf.float32)
        r = K.concatenate([x, zeros_w], 1)

        s = K.shape(r)
        zeros_h = tf.zeros((s[0], s[1], 1, s[3]), tf.float32)
        r = K.concatenate([r, zeros_h], 2)
        return r    

    def compute_output_shape(self, input_shape):
        shape = tf.TensorShape(input_shape).as_list()
        if K.image_data_format() == 'channels_last':
            shape[1] = shape[1] + 1
            shape[2] = shape[2] + 1
        else:
            shape[2] = shape[2] + 1
            shape[3] = shape[3] + 1
        return tf.TensorShape(shape)

# Loading Models

In [13]:
model_from_file = tf.keras.models.load_model('dlib_face_recognition_resnet_model_v1.h5',
    custom_objects={'ScaleLayer': ScaleLayer, 'ReshapeLayer': ReshapeLayer})
pose_predictor = dlib.shape_predictor('shape_predictor_5_face_landmarks.dat')
face_encoder = dlib.face_recognition_model_v1('dlib_face_recognition_resnet_model_v1.dat')

dir='faces/'
col=imread_collection(dir+'/*.jpg')

known_faces_dir='known_faces/'

c:\Users\karan\AppData\Local\Programs\Python\Python37\lib\site-packages\tensorflow\python\keras\layers\core.py:1059: UserWarning: converter.model is not loaded, but a Lambda layer uses it. It may cause errors.
  , UserWarning)


# Defning helper functions

In [14]:
def encodings(img,pose_predictor,face_encoder):
    face_locations=[dlib.rectangle(left=0, top=0, right=img.shape[1], bottom=img.shape[0])]
    predictors = [pose_predictor(img, face_location) for face_location in face_locations]
    return [np.array(face_encoder.compute_face_descriptor(img, predictor, 1)) for predictor in predictors]
def get_embedding(img):
    try:
        embedding = encodings(img,pose_predictor,face_encoder)[0]
        return embedding
    except:
        return None
def distance_based_compare(to_check_from,checker):
    if len(to_check_from)==0:
        return np.empty((0))
    result=np.linalg.norm(to_check_from - checker, axis=1)
    return list(result<=0.5)
def get_embedding_tf(img):
    try:
        embedding=model_from_file.predict(img, batch_size=1)[0]
        return embedding
    except:
        return None
def convert_image_to_np(img):
    return np.asarray(img, dtype='float32')
def normalize_image(image):    
    [R,G,B] = np.dsplit(image,image.shape[-1])
    Rx = (R - 122.782) / 256.
    Gx = (G - 117.001) / 256.
    Bx = (B - 104.298) / 256.

    new_image = np.dstack((Rx,Gx,Bx))
    return new_image

## DLIB embeddings

In [24]:
vars=[get_embedding(i) for i in col]

## Tensorflow embeddings

In [25]:
np_vars=[np.asarray(i, dtype='float32') for i in col]
normalized_np_vars=[normalize_image(i) for i in np_vars]
final_vars = list(map(lambda x: np.reshape(np.resize(x,(150,150,3)),(1,150,150,3)), normalized_np_vars))
vars1=[get_embedding_tf(i) for i in final_vars]

### This shows the average euclidean distance between Tensorflow's model and dlib's model which is 0.78

In [26]:
ctr=0
for i,j in zip(vars,vars1):
    ctr+=np.linalg.norm([i]-j,axis=1)
print(ctr/len(vars1))

[0.78247926]


### Now comparing the mean embeddings of a supervised data 

In [19]:
ctr=0
for i in os.listdir(known_faces_dir):
    known_col=imread_collection(known_faces_dir+i+'/*.jpg')
    np_vars=[np.asarray(i, dtype='float32') for i in known_col]
    normalized_np_vars=[normalize_image(i) for i in np_vars]
    final_vars = list(map(lambda x: np.reshape(np.resize(x,(150,150,3)),(1,150,150,3)), normalized_np_vars))
    vars1=[get_embedding_tf(i) for i in final_vars]
    value=[np.linalg.norm([vars1[1]]-i,axis=1) for i in vars1]
    avg=sum(value)/len(vars1)
    ctr+=avg
print(ctr/2)

[0.30099833]


In [20]:
ctr1=0
for i in os.listdir(known_faces_dir):
    known_col=imread_collection(known_faces_dir+i+'/*.jpg')
    vars=[get_embedding(i) for i in known_col]
    value=[np.linalg.norm([vars[1]]-i,axis=1) for i in vars]
    avg=sum(value)/len(value)
    ctr1+=avg
print(ctr1/2)

[0.47880862]
